In [0]:
storage_account = "stdataengdevrazi01"
storage_key = "gh/fNSxGWb7Ap18fL8XDOYA93goBm0I1gKrFHow8KjUllgGJfdSLl9Kv7w8uKKLVvunnW1/UWgbC+AStyOvmUA=="

# Read Silver Delta layer
silver_path = f"abfss://data@{storage_account}.dfs.core.windows.net/silver/customers_silver"

df_silver = spark.read \
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key) \
    .format("delta") \
    .load(silver_path)

display(df_silver)

In [0]:
from pyspark.sql.functions import count, current_timestamp

# Aggregate total customer registrations per city
df_gold_city_summary = df_silver \
    .groupBy("City") \
    .agg(count("CustomerID").alias("TotalCustomers")) \
    .withColumn("CalculatedAt", current_timestamp())

display(df_gold_city_summary)

In [0]:
gold_path = f"abfss://data@{storage_account}.dfs.core.windows.net/gold/dim_customer_city_summary"

# Save aggregated table in Gold layer
df_gold_city_summary.write \
    .option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key) \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

print("Silver to Gold layer aggregation completed successfully!")